In [10]:
import os
from pathlib import Path

import pandas as pd
from docx import Document
import PyPDF2
from loguru import logger


def extract_text_from_pdf(file_path):
    text = ""
    with open(file_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text


def extract_text_from_docx(file_path):
    doc = Document(file_path)
    text = "\n".join([p.text for p in doc.paragraphs])
    return text


def extract_text_from_excel(file_path):
    df = pd.read_excel(file_path, sheet_name=None)
    text = ""
    for sheet_name, sheet in df.items():
        text += f"Sheet: {sheet_name}\n"
        text += sheet.astype(str).apply(lambda row: ' | '.join(row), axis=1).str.cat(sep="\n")
        text += "\n"
    return text


def extract_text_from_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def collect_documents(folder_path):
    data = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            ext = Path(file_path).suffix.lower()
            try:
                if ext == ".pdf":
                    text = extract_text_from_pdf(file_path)
                elif ext == ".docx":
                    text = extract_text_from_docx(file_path)
                elif ext in [".xls", ".xlsx"]:
                    text = extract_text_from_excel(file_path)
                elif ext == ".txt":
                    text = extract_text_from_txt(file_path)
                else:
                    continue
                
                data.append({
                    "file_path": file_path,
                    "file_name": file,
                    "extension": ext,
                    "content": text
                })
            except Exception as e:
                logger.error(f"{file_path}: {e}")
    return data

folder_path = Path("/home/yugoff/Downloads/projects/my/laboration/project-lab-sp/data").resolve()
documents_data = collect_documents(folder_path)

output_path = Path("/home/yugoff/Downloads/projects/my/laboration/project-lab-sp/scripts/full_directory").resolve()
output_file = output_path / "documents_data.csv"
df = pd.DataFrame(documents_data)
df.to_csv(output_file, index=False, encoding="utf-8")
logger.info(f"Завершили подготовку данных, все храним в {os.path.join(os.getcwd(), output_path)}!")


2026-02-09 15:57:36.442 | INFO     | __main__:<module>:75 - Завершили подготовку данных, все храним в /home/yugoff/Downloads/projects/my/laboration/project-lab-sp/scripts/full_directory!


In [11]:
import pandas as pd


df = pd.read_csv(output_file)
df_dict = df.to_dict(orient="records")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

df["clean_text"] = (
    df["content"].apply(clean_text)
)


In [12]:
df_content_to_list = df["clean_text"].to_list()
df_file_path_to_list = df["file_path"].to_list()

chunks = []
num = 0
for path, doc in zip(df_file_path_to_list, df_content_to_list):
    words = doc.split()
    for i in range(0, len(words), 180):
        chunk = " ".join(words[i : i + 200])
        chunks.append({path: chunk})
    num += 1

In [13]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
all_embeddings = []
meta_data = []

for item in chunks:
    for path, text in item.items():
        embedding = encoder.encode(text)
        all_embeddings.append(embedding)
        meta_data.append({"path": path, "text": text})

np_embeddings = np.array(all_embeddings).astype(np.float32)
dimension = np_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np_embeddings)


/home/yugoff/Downloads/projects/my/laboration/project-lab-sp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
faiss.write_index(index, "vector.index")


In [15]:
import json


with open("meta_data.json", "w", encoding="utf-8") as f:
    json.dump(meta_data, f, ensure_ascii=False, indent=4)


In [18]:
import json


with open("meta_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    print(data[2])

{'path': '/home/yugoff/Downloads/projects/my/laboration/project-lab-sp/data/Инженерная_записка_определение_координат_БПЛА.docx', 'text': 'анализа изображений и сопоставления с тайловыми подложками, что позволит получать оперативные данные; - Гибкость и масштабируемость системы: Возможность адаптации системы для работы с различными типами объектов и условий съемки; Пример привязки снимка к подложке местности, слева фрагмент снимка, справа фрагмент подложки'}
